# Notebook 3: CNN Model for Human Activity Recognition

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

In [2]:
base = r"C:\Users\M Mayavan\OneDrive"
base = base + r"\Desktop\HAR_Project\UCI HAR Dataset"
X_train = np.load(base + r"\X_train.npy")
X_test = np.load(base + r"\X_test.npy")
y_train = np.load(base + r"\y_train.npy")
y_test = np.load(base + r"\y_test.npy")

In [3]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [4]:
mean = X_train_t.mean(dim=(1, 2), keepdim=True)
std = X_train_t.std(dim=(1, 2), keepdim=True) + 1e-8
X_train_t = (X_train_t - mean) / std
mean = X_test_t.mean(dim=(1, 2), keepdim=True)
std = X_test_t.std(dim=(1, 2), keepdim=True) + 1e-8
X_test_t = (X_test_t - mean) / std

In [5]:
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv1d(9, 32, kernel_size=9, padding=4)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.7)
        self.fc1 = nn.Linear(2048, 64)
        self.fc2 = nn.Linear(64, 6)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.drop(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [7]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

In [8]:
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch + 1, total_loss)

1 60.37361494451761
2 16.31039149314165
3 13.465086203068495
4 13.089707743376493
5 11.830343652516603
6 11.415262432768941
7 11.400189591571689
8 10.083965940400958
9 9.503730310127139
10 8.856810679659247
11 9.166728580370545
12 8.95819694455713
13 7.496745457639918
14 7.5890581496059895
15 7.095239402260631
16 8.225438066758215
17 6.4559214767068624
18 6.999636865220964
19 6.883697272511199
20 8.45371578168124
21 6.166133447550237
22 6.093690481502563
23 5.929087359923869
24 6.232962815091014
25 5.717870227526873
26 5.7414571000263095
27 4.767428906867281
28 6.091302236076444
29 5.076019392814487
30 4.399526171036996
31 4.0438876217231154
32 4.2392869249451905
33 3.8833485203795135
34 3.7123540038010105
35 4.2595232027815655
36 4.937051506713033
37 4.0115288097877055
38 4.547708257683553
39 3.603152862167917
40 3.8939847638830543
41 3.6682526428776328
42 4.065282608557027
43 3.959499800344929
44 3.1929894315544516
45 3.711249881074764
46 3.1375391291221604
47 3.8484748182818294
48 3

In [9]:
model.eval()
train_preds = []
with torch.no_grad():
    for X_batch, y_batch in train_loader:
        out = model(X_batch)
        p = torch.argmax(out, dim=1)
        train_preds.append(p)
train_preds = torch.cat(train_preds)
correct = (train_preds == y_train_t).sum().item()
print(correct / len(y_train_t))

0.1779107725788901


In [10]:
model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = torch.argmax(output, dim=1)
        all_preds.append(preds)
all_preds = torch.cat(all_preds)
print(classification_report(y_test_t, all_preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       496
           1       0.95      0.94      0.94       471
           2       0.96      1.00      0.98       420
           3       0.85      0.77      0.81       491
           4       0.84      0.89      0.87       532
           5       1.00      1.00      1.00       537

    accuracy                           0.93      2947
   macro avg       0.93      0.93      0.93      2947
weighted avg       0.93      0.93      0.93      2947

